## notebook for analysis used in manuscript "Assessing the Accuracy of the NeuroBooster Array for GBA1 Variant Detection"

Author: Marco Toffoli (m.toffoli@ucl.ac.uk)

Last modified 08th September 2026

GP2 ❤️ Open Science 😍

## Setting paths

In [ ]:
# libraries and general paths
import pandas as pd
import numpy as np
import pgenlib
from pathlib import Path

# Gene of interest — used to filter probes from the .pvar files
GENE_OF_INTEREST = "GBA"

# Column names in the truth CSV
SAMPLE_ID_COL = "GP2ID"
TRUTH_CALL_COL = "GBA"

print("Done importing libraries")

## PD-Frotnline (PDF)

In [ ]:
# Truth CSV
RAP_CSV = Path("RAPSODIDNA-KitsAndSequencing_DATA_LABELS_2026-06-10_1747.csv")

# Importing PDF data

rap_raw = pd.read_csv(RAP_CSV, usecols=["RAPSODI ID", "GBA Allele 1", "GBA Allele 2"])
rap_raw = rap_raw.dropna(subset=["GBA Allele 1", "GBA Allele 2"], how="all")
rap_raw[["GBA Allele 1", "GBA Allele 2"]] = rap_raw[["GBA Allele 1", "GBA Allele 2"]].replace(["WT", "Ile153=","I158="], "0")

def merge_gba(row):
    g1 = str(row["GBA Allele 1"]).strip() if pd.notna(row["GBA Allele 1"]) else "0"
    g2 = str(row["GBA Allele 2"]).strip() if pd.notna(row["GBA Allele 2"]) else "0"
    if g1 != "0" and g2 != "0":
        return f"{g1}/{g2}"
    elif g1 != "0":
        return g1
    elif g2 != "0":
        return g2
    else:
        return "0"

rap_raw["GBA"] = rap_raw.apply(merge_gba, axis=1)

master_key = pd.read_csv("RAPSODI_master_key_n961.txt", sep="\t", usecols=["sample_id", "GP2ID"])
rap_raw = rap_raw.merge(master_key, left_on="RAPSODI ID", right_on="sample_id", how="left")
rap_raw["cohort"] = "PD-Frontline"

print("PD-Frontline importing complete\n")
print(f"Raw PD-Frontline file: {len(rap_raw)} rows")
print(f"  - Rows with null GP2ID: {rap_raw[SAMPLE_ID_COL].isna().sum()}")
print(f"  - Rows with null GBA:   {rap_raw[TRUTH_CALL_COL].isna().sum()}")

rap = rap_raw.dropna(subset=[SAMPLE_ID_COL, TRUTH_CALL_COL]).copy()
print(f"\nAfter dropping null GP2ID or GBA: {len(rap)} rows")

rap["GBA_carrier"] = (
    rap[TRUTH_CALL_COL].astype(str).str.strip().ne("0")
).astype(int)

print(f"\nCarrier breakdown in PD-Frontline set:")
print(rap["GBA_carrier"].value_counts().rename({0: "Non-carrier (0)", 1: "Carrier (1)"}))
print(f"\nDistinct GBA calls present:")
print(rap[TRUTH_CALL_COL].value_counts().to_string())

# Keep a clean working copy
rap = rap.reset_index(drop=True)
print(f"\nPD-Frontline dataframe ready: {rap.shape}")
rap.head()

## PPMI

In [ ]:
# Truth CSV
PPMI_CSV = Path("PPMI_export_final_no_missing.csv")

PPMI_raw = pd.read_csv(PPMI_CSV, usecols=[SAMPLE_ID_COL, TRUTH_CALL_COL])
PPMI_raw["cohort"] = "PPMI"

print("PPMI importing complete\n")
print(f"Raw PPMI file: {len(PPMI_raw)} rows")
print(f"  - Rows with null GP2ID: {PPMI_raw[SAMPLE_ID_COL].isna().sum()}")
print(f"  - Rows with null GBA:   {PPMI_raw[TRUTH_CALL_COL].isna().sum()}")

PPMI = PPMI_raw.dropna(subset=[SAMPLE_ID_COL, TRUTH_CALL_COL]).copy()
print(f"\nAfter dropping null GP2ID or GBA: {len(PPMI)} rows")

PPMI["GBA_carrier"] = (
    PPMI[TRUTH_CALL_COL].astype(str).str.strip().ne("0")
).astype(int)

print(f"\nCarrier breakdown in PPMI set:")
print(PPMI["GBA_carrier"].value_counts().rename({0: "Non-carrier (0)", 1: "Carrier (1)"}))
print(f"\nDistinct GBA calls present:")
print(PPMI[TRUTH_CALL_COL].value_counts().to_string())

# clean working copy
PPMI = PPMI.reset_index(drop=True)
print(f"\nPPMI dataframe ready: {PPMI.shape}")

## LongNext

In [ ]:
# Truth CSV
LN_CSV = Path("GBA_variants_Pavia_NBA_toffoli.csv")

LN_imp = pd.read_csv(LN_CSV, usecols=[SAMPLE_ID_COL, "GBA1_HGVSc-HGVSp", "Analysis"])
LN_imp = LN_imp.rename(columns={"GBA1_HGVSc-HGVSp": TRUTH_CALL_COL})
LN_raw = LN_imp[LN_imp["Analysis"] == "LongNext"].copy()
LN_raw[TRUTH_CALL_COL] = LN_raw[TRUTH_CALL_COL].fillna("0").replace("", "0")
LN_raw["cohort"] = "LongNext"

print("LongNext importing complete\n")
print(f"Raw LongNext file: {len(LN_raw)} rows")
print(f"  - Rows with null GP2ID: {LN_raw[SAMPLE_ID_COL].isna().sum()}")
print(f"  - Rows with null GBA:   {LN_raw[TRUTH_CALL_COL].isna().sum()}")

LN = LN_raw.dropna(subset=[SAMPLE_ID_COL, TRUTH_CALL_COL]).copy()
print(f"\nAfter dropping null GP2ID or GBA: {len(LN)} rows")

LN["GBA_carrier"] = (
    LN[TRUTH_CALL_COL].astype(str).str.strip().ne("0")
).astype(int)

print(f"\nCarrier breakdown in LongNext set:")
print(LN["GBA_carrier"].value_counts().rename({0: "Non-carrier (0)", 1: "Carrier (1)"}))
print(f"\nDistinct GBA calls present:")
print(LN[TRUTH_CALL_COL].value_counts().to_string())

# clean working copy
LN = LN.reset_index(drop=True)
print(f"\nLongNext dataframe ready: {LN.shape}")

## merging everything together

In [ ]:
# Keep only relevant columns from each
ppmi_sub = PPMI[["GP2ID", "GBA", "cohort", "GBA_carrier"]]
rap_sub = rap[["GP2ID", "GBA", "cohort", "GBA_carrier"]]
ln_sub = LN[["GP2ID", "GBA", "cohort", "GBA_carrier"]]

# Stack them vertically
truth = pd.concat([ppmi_sub, rap_sub, ln_sub], ignore_index=True)

print("Merging datasets complete")

## filtering plink files for GBA1 only

In [ ]:
from pathlib import Path

RAW_DIR = Path("workspace/gp2_tier2_eu_release11/raw_genotypes")

wanted = set(truth[SAMPLE_ID_COL].dropna())

keep_rows = []
for psam_path in sorted(RAW_DIR.glob("*/*.psam")):
    psam = pd.read_csv(psam_path, sep="\t")
    psam.columns = [c.lstrip("#") for c in psam.columns]      # "#FID"/"#IID" -> "FID"/"IID"
    id_col  = "IID" if "IID" in psam.columns else psam.columns[0]
    hit = psam[psam[id_col].isin(wanted)]
    if "FID" in psam.columns:
        keep_rows.append(hit[["FID", id_col]].rename(columns={id_col: "IID"}))
    else:                                          # psam has no FID -> plink2 keys on IID
        keep_rows.append(pd.DataFrame({"FID": hit[id_col], "IID": hit[id_col]}))

keep = pd.concat(keep_rows, ignore_index=True).drop_duplicates()

# header starts with "#" so plink2 reads it as a real FID/IID header
keep.to_csv("gba_truth_samples.txt", sep="\t", index=False, header=["#FID", "IID"])

# diagnostics
n_diff    = (keep["FID"].astype(str) != keep["IID"].astype(str)).sum()
not_found = wanted - set(keep["IID"])
print(f"Keep list: {len(keep)} rows, {keep['IID'].nunique()} unique IIDs")
print(f"Wanted GP2IDs not present in ANY .psam: {len(not_found)}")

In [ ]:
%%bash
set -euo pipefail

PLINK_DIR="workspace/gp2_tier2_eu_release11/raw_genotypes"
OUTPUT_PATH="filtered_plink2"
SAMPLE_LIST="gba_truth_samples.txt"
mkdir -p "$OUTPUT_PATH"

for f in "$PLINK_DIR"/*/*.pgen
do
    base=$(basename "$f" .pgen)
    pfile="${f%.pgen}"

    # Genome-wide missingness QC. Some ancestry filesets contain none of our
    # samples; plink2 then exits non-zero, so allow that one step to "fail"
    # and skip the fileset instead of aborting the whole loop.
    plink2 \
        --pfile "$pfile" \
        --keep "$SAMPLE_LIST" \
        --mind 0.05 \
        --write-samples \
        --out "$OUTPUT_PATH/${base}_qcpass" || true

    if [[ ! -s "$OUTPUT_PATH/${base}_qcpass.id" ]]; then
        echo ">> ${base}: no matching samples — skipped"
        continue
    fi
    echo ">> ${base}: $(grep -vc '^#' "$OUTPUT_PATH/${base}_qcpass.id") samples passed QC"

    # Extract the GBA region for samples that passed genome-wide QC
    plink2 \
        --pfile "$pfile" \
        --keep "$OUTPUT_PATH/${base}_qcpass.id" \
        --chr 1 \
        --from-bp 155234452 \
        --to-bp 155244627 \
        --make-pgen \
        --out "$OUTPUT_PATH/${base}"
done

echo "=== total samples kept across all filesets ==="
cat "$OUTPUT_PATH"/*_qcpass.id 2>/dev/null | grep -vc '^#'

## Filter truth to samples present in at least one plink2 fileset

In [ ]:
# Collect all sample IDs across all filtered plink2 filesets
all_genotyped_samples = set()

for pgen_path in sorted(Path("filtered_plink2").glob("*.pgen")):
    prefix = pgen_path.with_suffix("")
    psam   = pd.read_csv(prefix.with_suffix(".psam"), sep="\t")
    psam.columns = [c.lstrip("#") for c in psam.columns]
    id_col = "IID" if "IID" in psam.columns else psam.columns[0]
    all_genotyped_samples.update(psam[id_col].tolist())

print(f"Total unique genotyped samples across all filesets: {len(all_genotyped_samples)}")

# Filter truth
truth_before = truth.groupby("cohort").size()
truth = truth[truth[SAMPLE_ID_COL].isin(all_genotyped_samples)].reset_index(drop=True)
truth_after = truth.groupby("cohort").size()


print("Truth samples before/after filtering by cohort:")
print(f"{'Cohort':<20} {'Before':>8} {'After':>8} {'Removed':>8}")
print("-" * 46)

total_before = total_after = 0
for cohort in truth_before.index:
    before = truth_before.get(cohort, 0)
    after = truth_after.get(cohort, 0)
    total_before += before
    total_after += after
    print(f"{cohort:<20} {before:>8} {after:>8} {before-after:>8}")

print("-" * 46)
print(f"{'TOTAL':<20} {total_before:>8} {total_after:>8} {total_before-total_after:>8}")

print(f"\nCarrier breakdown after filtering (by cohort):")
print(
    truth.groupby("cohort")["GBA_carrier"]
    .value_counts()
    .rename({0: "Non-carrier", 1: "Carrier"})
    .to_string()
)

# Overall carrier breakdown across all cohorts
print(f"\nCarrier breakdown after filtering (all cohorts combined):")
print(
    truth["GBA_carrier"]
    .value_counts()
    .rename({0: "Non-carrier", 1: "Carrier"})
    .sort_index()
    .to_string()
)

print(f"\nDistinct GBA calls after filtering:")
print(truth[TRUTH_CALL_COL].value_counts().to_string())

## Building table to cross-reference variants (and checking with one pvar file)

In [15]:
#renaming some truth calls for consistency
ALIASES = {
    "A495P Val499= L483P":           "RecNciI",
    "84GG":                          "L29Afs*18",
    "IVS9+1":                        "c.1388+1G>A",
    "S212Ter":                       "S212X",
    "IVS2+1G>A":                     "c.115+1G>A",
    "IVS2+1":                        "c.115+1G>A",
    "NM_000157.4(GBA):c.115+1G>A":   "c.115+1G>A",
}

def normalize_gba(call):
    if pd.isna(call):
        return call
    toks = [ALIASES.get(t.strip(), t.strip()) for t in str(call).split("/")]
    toks = [t for t in toks if t not in ("", "nan", "0")]
    return "/".join(toks) if toks else "0"

truth[TRUTH_CALL_COL] = truth[TRUTH_CALL_COL].map(normalize_gba)

In [ ]:
variant_map = pd.DataFrame([
    # name          pos         rsID
    ("R502C",       155235196,  "rs80356771"),
    ("E365K",       155236376,  "rs2230288"),
    ("T408M",       155236246,  "rs75548401"),
    ("S212X",       155238260,  "rs1671872221"),
    ("N409S",       155235843,  "rs76763715"),
    ("rs3115534-G", 155235878,  "rs3115534"),
    ("L483P",       155235252,  "rs421016"),
    ("R159W",       155235756,  "rs439898"),
    ("L29Afs*18",   155240660,  "rs387906315"),
    ("R535H",       155235002,  "rs75822236"),
    ("G232E",       155238200,  "rs1376479747"),
    ("D179H",       155238570,  "rs147138516"),
    ("F255Y",       155237576,  "rs74500255"),
    ("Q112X",       155239736,  "rs1671974195"),
    ("A495P",       155235217,  "rs368060"),
    ("W223R",       155238228,  "rs61748906"),
    ("D482N",       155235256,  "rs75671029"),
    ("E427K",       155235790,  "rs149171124"),
    ("M400I",       155236269,  "rs149487315"),
    ("W396C",       155236282,  "W396C"),
    ("G234E",       155238194,  "rs74462743"),
    ("P217S",       155238246,  "rs2524836722"),
    ("R170L",       155238596,  "rs80356763"),
    ("R87W",        155239934,  "rs1141814"),
    ("R83C",        155239946,  "rs1141812"),
    ("R78C",        155239961,  "rs146774384"),
    ("C62Y",        155240008,  "rs145888253"),
    ("c.762-2A>G",  155237580,  "rs2148075664"),
    ("c.1388+1G>A", 155235680,  "rs1671699033"),
    ("V433L",       155235772,  "rs80356769"),
    ("K196Q",       155238519,  "rs121908297"),
    ("G49S",        155240048,  "rs760930573"),
    ("T306I",       155237423,  "rs199628072"),
    ("R86Q",        155239937,  "rs1671987417"),
    ("G241R",       155238174,  "rs409652"),
    ("D448H",       155235727,  "rs1064651"),
    ("G103S",       155239762,  "rs748485792"),
    ("c.115+1G>A",  155240629,  "rs104886460"),
    ("R368C",       155236367,  "rs374306700"),
    ("G154R",       155238645,  "rs769343650"),
    ("R78H",        155239960,  "rs752857428"),
    ("N58S",        155240020,  "rs74797197"),
    ("D419N",       155235814,  "D419N"),
    ("R170C",       155238597,  "rs398123530"),
    ("S149P",       155239624,  "c.445T>C"),
    ("M319T",       155237384,  "c.956T>C"),
    ("V499L",       155235205,  "rs369068553"),
    ("E365D",       155236374,  "rs80317710"),
    ("F252I",       155238141,  "rs381737"),
    ("R296X",       155237454,  "rs1553217626"),
    ("L500P",       155235201,  "rs1362103320"),
    ("c.1505+1G>T", 155235194,  "c.1505+1G>T"),
    ("K13R",        155240707,  "rs150466109"),
    ("L29Afs*18",   155240662,  "rs387906315"),
    ("R86Q",        155239936,  "rs144173415"),
    ("T336S",       155236463,  "rs1553217322")
    
],
columns=["variant_name", "pos_grch38", "rsID"])

print("=== Variant lookup table ===")
print(variant_map.to_string(index=False))

## loading genotypes from filtered plink2 files

In [ ]:
pos_to_variant = dict(
    zip(variant_map["pos_grch38"].dropna().astype(int),
        variant_map["variant_name"])
)

# Scan ALL pvar files to build complete probe inventory
print("=== Scanning all .pvar files for GBA probes ===")
all_gba_probes = []

for pgen_path in sorted(Path("filtered_plink2").glob("*.pgen")):
    prefix = pgen_path.with_suffix("")
    pvar   = pd.read_csv(prefix.with_suffix(".pvar"), sep="\t")
    pvar.columns = [c.lstrip("#") for c in pvar.columns]
    pvar["variant_name"] = pvar["POS"].astype(int).map(pos_to_variant)
    pvar["source_file"] = prefix.name
    all_gba_probes.append(pvar)
    print(f"  {prefix.name}: {len(pvar)} GBA probes "
          f"({pvar['variant_name'].nunique()} variants)")

all_gba_probes_df = pd.concat(all_gba_probes, ignore_index=True)

print(f"\n=== Complete probe inventory across all filesets ===")
print(all_gba_probes_df.groupby("variant_name")["ID"]
      .apply(lambda x: sorted(x.unique().tolist()))
      .to_string())
print(f"\nTotal unique probes: {all_gba_probes_df[['ID','POS']].drop_duplicates().shape[0]}")
print(f"Total unique positions covered: {all_gba_probes_df['POS'].nunique()}")

# Load genotypes from each fileset
print("\n=== Loading genotypes ===")
all_geno_dfs = []

for pgen_path in sorted(Path("filtered_plink2").glob("*.pgen")):
    prefix = pgen_path.with_suffix("")
    pvar   = pd.read_csv(prefix.with_suffix(".pvar"), sep="\t")
    psam   = pd.read_csv(prefix.with_suffix(".psam"), sep="\t")
    pvar.columns = [c.lstrip("#") for c in pvar.columns]
    psam.columns = [c.lstrip("#") for c in psam.columns]

    id_col     = "IID" if "IID" in psam.columns else psam.columns[0]
    sample_ids = psam[id_col].tolist()
    n_samples  = len(sample_ids)

    pvar["variant_name"] = pvar["POS"].astype(int).map(pos_to_variant)

    if pvar.empty:
        print(f"  {prefix.name}: no GBA probes — skipping")
        continue

    #  Use original pvar index as pgen variant index 
    original_indices = pvar.index.tolist()  # these match pgen row order

    with pgenlib.PgenReader(bytes(pgen_path)) as reader:
        geno_matrix = np.empty((len(pvar), n_samples), dtype=np.int32)
        for i, var_idx in enumerate(original_indices):
            reader.read(var_idx, geno_matrix[i])

    #  Reshape to long format 
    rows = []
    for i, (_, probe_row) in enumerate(pvar.iterrows()):
        for j, sample_id in enumerate(sample_ids):
            rows.append({
                "sample_id":    sample_id,
                "probe_id":     probe_row["ID"],
                "pos":          probe_row["POS"],
                "variant_name": probe_row["variant_name"],
                "array_call":   geno_matrix[i, j],
            })

    df = pd.DataFrame(rows)
    df["source_file"] = prefix.name
    all_geno_dfs.append(df)
    print(f"  {prefix.name}: {len(pvar)} probes × {n_samples} samples loaded")

# Concatenate and clean 
genotypes = pd.concat(all_geno_dfs, ignore_index=True)
genotypes["array_call"] = genotypes["array_call"].replace(-9, np.nan).astype(pd.Int32Dtype())

print(f"\nGenotypes dataframe shape: {genotypes.shape}")
print(f"Unique probes:   {genotypes['probe_id'].nunique()}")
print(f"Unique samples:  {genotypes['sample_id'].nunique()}")
print(f"\nArray call distribution (0=hom ref, 1=het, 2=hom alt, NaN=missing):")
print(genotypes["array_call"].value_counts(dropna=False))

## Merge genotypes with truth set

In [ ]:
# Parse each truth GBA call into a set of variant tokens
def parse_truth_tokens(gba_call):
    if pd.isna(gba_call):
        return set()
    s = str(gba_call).strip()
    if s in ("0", "", "nan"):
        return set()
    return {t.strip() for t in s.split("/") if t.strip() not in ("", "0")}

# One truth row per sample (warn if a GP2ID shows up more than once)
dups = truth[SAMPLE_ID_COL].duplicated(keep=False)
if dups.any():
    print(f"WARNING: {truth.loc[dups, SAMPLE_ID_COL].nunique()} sample(s) in >1 truth row:")
    print(truth.loc[dups, [SAMPLE_ID_COL, "cohort", TRUTH_CALL_COL]]
          .sort_values(SAMPLE_ID_COL).to_string(index=False))

truth_u = truth.drop_duplicates(subset=[SAMPLE_ID_COL], keep="first").copy()
truth_u["truth_tokens"] = truth_u[TRUTH_CALL_COL].map(parse_truth_tokens)

# Keep only probes we can label (known variant_name)
geno = genotypes.dropna(subset=["variant_name"]).copy()

# Collapse any duplicate (sample, probe) rows, preferring a non-missing call
geno = (geno.sort_values("array_call", na_position="last")
            .drop_duplicates(subset=["sample_id", "probe_id"], keep="first"))

# Join array calls to truth (inner = samples with BOTH array + truth)
comp = geno.merge(
    truth_u[[SAMPLE_ID_COL, "cohort", TRUTH_CALL_COL, "truth_tokens"]],
    left_on="sample_id", right_on=SAMPLE_ID_COL, how="inner"
)

# Label each call
comp["array_missing"] = comp["array_call"].isna()
comp["truth_pos"] = comp.apply(
    lambda r: int(r["variant_name"] in r["truth_tokens"]), axis=1
)

valid = comp[~comp["array_missing"]].copy()
valid["array_pos"] = (valid["array_call"].astype(int) >= 1).astype(int)

conditions = [
    (valid.truth_pos == 1) & (valid.array_pos == 1),  # TP
    (valid.truth_pos == 0) & (valid.array_pos == 1),  # FP
    (valid.truth_pos == 1) & (valid.array_pos == 0),  # FN
    (valid.truth_pos == 0) & (valid.array_pos == 0),  # TN
]
valid["call_class"] = np.select(conditions, ["TP", "FP", "FN", "TN"], default="?")

print(f"Evaluable (sample × probe) calls: {len(valid):,}")
print(f"Dropped — missing array call:     {comp['array_missing'].sum():,}")
print(f"Samples evaluated:                {valid['sample_id'].nunique():,}")
print("\nCall-class totals:")
print(valid["call_class"].value_counts().to_string())

## Per probe metrics

In [ ]:
counts = (valid.groupby(["variant_name", "probe_id", "pos"])["call_class"]
               .value_counts().unstack(fill_value=0)
               .reindex(columns=["TP", "FP", "TN", "FN"], fill_value=0)
               .reset_index())

# Add missing-call count per probe
miss = (comp[comp["array_missing"]]
        .groupby("probe_id").size().rename("missing").reset_index())
counts = counts.merge(miss, on="probe_id", how="left")
counts["missing"] = counts["missing"].fillna(0).astype(int)

# Attach REF/ALT
probe_alleles = (all_gba_probes_df.rename(columns={"ID": "probe_id"})
                 [["probe_id", "REF", "ALT"]].drop_duplicates("probe_id"))
counts = counts.merge(probe_alleles, on="probe_id", how="left")

def rate(num, den):
    return np.where(den > 0, num / den, np.nan)

counts["n_eval"]       = counts[["TP", "FP", "TN", "FN"]].sum(axis=1)
counts["n_truth_pos"]  = counts.TP + counts.FN
counts["sensitivity"]  = rate(counts.TP, counts.TP + counts.FN)   # = recall
counts["specificity"]  = rate(counts.TN, counts.TN + counts.FP)
counts["precision"]    = rate(counts.TP, counts.TP + counts.FP)   # PPV
counts["recall"]       = counts["sensitivity"]

cols = ["variant_name", "probe_id", "pos", "REF", "ALT",
        "TP", "FP", "TN", "FN", "missing", "n_truth_pos",
        "sensitivity", "specificity", "precision", "recall"]
probe_metrics = counts[cols].sort_values(["n_truth_pos", "variant_name"],
                                         ascending=[False, True]).reset_index(drop=True)

pd.set_option("display.max_rows", None)
print(probe_metrics.round(3).to_string(index=False))
probe_metrics.to_csv("gba_probe_metrics.csv", index=False)
print("\nSaved → gba_probe_metrics.csv")

## Pooled metrics

In [ ]:
keep = ["sample_id", "cohort", "variant_name", "probe_id", "pos",
        "array_call", TRUTH_CALL_COL]

fp_calls = valid[valid.call_class == "FP"][keep].sort_values(["variant_name", "sample_id"])
fn_calls = valid[valid.call_class == "FN"][keep].sort_values(["variant_name", "sample_id"])

print(f"False positives: {len(fp_calls)}   |   False negatives: {len(fn_calls)}")
fp_calls.to_csv("gba_false_positives.csv", index=False)
fn_calls.to_csv("gba_false_negatives.csv", index=False)

# Pooled (micro-average) metrics across all probes
tot = valid["call_class"].value_counts()
TP, FP, TN, FN = (int(tot.get(k, 0)) for k in ["TP", "FP", "TN", "FN"])
print("\n=== Pooled across all probes ===")
print(f"TP={TP}  FP={FP}  TN={TN}  FN={FN}")
print(f"Sensitivity (recall): {TP/(TP+FN):.3f}" if TP+FN else "Sensitivity: n/a")
print(f"Specificity:          {TN/(TN+FP):.3f}" if TN+FP else "Specificity: n/a")
print(f"Precision (PPV):      {TP/(TP+FP):.3f}" if TP+FP else "Precision: n/a")

# Truth variants with NO matching probe (not evaluable)
probe_vars  = set(geno["variant_name"].dropna().unique())
truth_vars  = set().union(*truth_u["truth_tokens"]) if len(truth_u) else set()
unmatched   = sorted(truth_vars - probe_vars)
print(f"\nTruth variants present but with no probe ({len(unmatched)}): {unmatched}")

## Sample phenotypes by cohort (GP2 release-11 master key)

In [ ]:
# Attach the GP2 release-11 phenotype to every sample and tabulate it

GP2_MASTER_KEY = "workspace/gp2_tier2_eu_release11/clinical_data/master_key_release11_final_vwb.csv"

mk = (pd.read_csv(GP2_MASTER_KEY, usecols=["GP2ID", "baseline_GP2_phenotype"])
        .drop_duplicates("GP2ID"))

# (re)attach the phenotype column to the one-row-per-sample truth table
truth_u = (truth_u.drop(columns=["baseline_GP2_phenotype"], errors="ignore")
                  .merge(mk, on=SAMPLE_ID_COL, how="left"))

n_missing = truth_u["baseline_GP2_phenotype"].isna().sum()
print(f"Samples with no phenotype in the master key: {n_missing} / {len(truth_u)}")
if n_missing:
    print(truth_u[truth_u["baseline_GP2_phenotype"].isna()]
          .groupby("cohort")[SAMPLE_ID_COL].nunique().rename("unmatched").to_string())

# Diagnosis figures for each cohort (one row per baseline_GP2_phenotype value)
pheno_tab = truth_u.copy()
pheno_tab["baseline_GP2_phenotype"] = pheno_tab["baseline_GP2_phenotype"].fillna("(no phenotype)")
pheno_counts = pd.crosstab(pheno_tab["baseline_GP2_phenotype"], pheno_tab["cohort"])
pheno_counts["TOTAL"] = pheno_counts.sum(axis=1)
pheno_counts = pheno_counts.sort_values("TOTAL", ascending=False)
pheno_counts.loc["ALL_samples"] = pheno_counts.sum()

print("\n=== baseline_GP2_phenotype counts by cohort ===")
print(pheno_counts.to_string())
pheno_counts.to_csv("gba_phenotype_counts_by_cohort.csv")
print("\nSaved -> gba_phenotype_counts_by_cohort.csv")

## Allele frequency table for paper

In [ ]:
PHENOTYPE_GROUP = {
    "PD":              "PD",
    "Affected_PD":     "PD",
    "Prodromal":       "Ctrl",
    "Control":         "Ctrl",
    "Unaffected":      "Ctrl",
    "Other":           "Other",
    "Affected_Other":  "Other",
    "(no phenotype)":  "exclude",
}

taf = truth_u.copy()
taf["baseline_GP2_phenotype"] = taf["baseline_GP2_phenotype"].fillna("(no phenotype)")

present  = set(taf["baseline_GP2_phenotype"].unique())
unmapped = sorted(p for p in present if p not in PHENOTYPE_GROUP)
if unmapped:
    print("These phenotypes are in your data but not in PHENOTYPE_GROUP — "
          "assign each to PD / Ctrl / Other / exclude, then re-run:")
    for p in unmapped:
        print("   ", p)
else:
    taf["group"] = taf["baseline_GP2_phenotype"].map(PHENOTYPE_GROUP)
    KEEP = [g for g in dict.fromkeys(PHENOTYPE_GROUP.values()) if g != "exclude"]
    taf = taf[taf["group"].isin(KEEP)].copy()

    def af_table(col_values, col_order):
        """Allele-frequency table for a given column grouping. Returns (pct_table, counts)."""
        d = taf.assign(af_col=col_values)
        al = d[[SAMPLE_ID_COL, "af_col", TRUTH_CALL_COL]].copy()
        al["tok"] = al[TRUTH_CALL_COL].astype(str).str.strip().str.split("/")
        al = al.explode("tok")
        al["tok"] = al["tok"].str.strip()
        al = al[~al["tok"].isin(["0", "", "nan"])]

        cnt = al.groupby(["tok", "af_col"]).size().unstack(fill_value=0)
        N   = d.groupby("af_col")[SAMPLE_ID_COL].nunique()
        cols = [c for c in col_order if c in N.index]      # keep order, drop empty combos
        cnt = cnt.reindex(columns=cols, fill_value=0)

        af = cnt.divide(2 * N[cols], axis=1)               # AF = alt alleles / 2N
        af["TOTAL"] = cnt.sum(axis=1) / (2 * N[cols].sum())
        af = af.sort_values("TOTAL", ascending=False)
        cnt = cnt.reindex(af.index)

        burden = pd.Series({c: cnt[c].sum() / (2 * N[c]) for c in cols}, name="ALL_variants")
        burden["TOTAL"] = cnt.sum().sum() / (2 * N[cols].sum())
        af = pd.concat([af, burden.to_frame().T])
        n_row = pd.Series({**N[cols].to_dict(), "TOTAL": int(N[cols].sum())}, name="N_samples")
        af = pd.concat([af, n_row.to_frame().T])

        pct = af.astype(object)
        m = pct.index != "N_samples"
        pct.loc[m] = (af.loc[m].astype(float) * 100).round(3).astype(str) + "%"
        pct.loc["N_samples"] = af.loc["N_samples"].astype(int)

        cnt_out = cnt.copy(); cnt_out["TOTAL"] = cnt_out.sum(axis=1); cnt_out.loc["ALL_variants"] = cnt_out.sum()
        return pct, cnt_out

    # cohort order for the per-cohort table
    cohort_order = [c for c in ["LongNext", "PD-Frontline", "PPMI"] if c in taf["cohort"].unique()]
    cohort_order += [c for c in taf["cohort"].unique() if c not in cohort_order]

    # Table A: pooled by phenotype group (PD / Ctrl / Other)
    pct_grp, cnt_grp = af_table(taf["group"], KEEP)
    print("=== Allele frequency (%) — by phenotype group (all cohorts pooled) ===")
    print(pct_grp.to_string())
    pct_grp.to_csv("gba_allele_freq_by_group.csv")
    cnt_grp.to_csv("gba_allele_counts_by_group.csv")

    # Table B: per cohort × phenotype group
    cxg_order = [f"{coh}-{grp}" for coh in cohort_order for grp in KEEP]
    pct_cxg, cnt_cxg = af_table(taf["cohort"] + "-" + taf["group"], cxg_order)
    print("\n=== Allele frequency (%) — by cohort × phenotype group ===")
    print(pct_cxg.to_string())
    pct_cxg.to_csv("gba_allele_freq_by_cohort_group.csv")
    cnt_cxg.to_csv("gba_allele_counts_by_cohort_group.csv")

    print("\nSaved -> gba_allele_freq_by_group.csv, gba_allele_freq_by_cohort_group.csv (+ *_counts_*)")

## Carriers by phenotype group and cohort

In [ ]:
# Carriers (person-level) by phenotype group and cohort
# A carrier is a praticipant with at least one non-reference GBA1 allele in the
# reference sequencing. Unlike the allele-frequency tables above (allele-level,
# where a homozygote counts twice), this counts people once. Groups match the
# allele-frequency cell: PD = PD + Affected_PD; Controls = Prodromal +
# Unaffected + Control; Other = Other + Affected_Other.

GROUP = {
    "PD": "PD", "Affected_PD": "PD",
    "Prodromal": "Controls", "Control": "Controls", "Unaffected": "Controls",
    "Other": "Other", "Affected_Other": "Other",
}

cc = truth_u.copy()
cc["baseline_GP2_phenotype"] = cc["baseline_GP2_phenotype"].fillna("(no phenotype)")
cc["group"] = cc["baseline_GP2_phenotype"].map(GROUP)

unmapped = sorted(set(cc.loc[cc["group"].isna(), "baseline_GP2_phenotype"]) - {"(no phenotype)"})
if unmapped:
    print("Unmapped phenotypes — add them to GROUP and re-run:", unmapped)
cc = cc[cc["group"].notna()].copy()          # drop (no phenotype) / unmapped

# carrier flag straight from the truth call, same token rule as the AF tables
def is_carrier(call):
    toks = [t.strip() for t in str(call).split("/")]
    return any(t not in ("0", "", "nan") for t in toks)
cc["carrier"] = cc[TRUTH_CALL_COL].map(is_carrier)

def carrier_tab(keys, order):
    d = cc.assign(k=keys)
    g = d.groupby("k").agg(carriers=("carrier", "sum"), N=(SAMPLE_ID_COL, "nunique"))
    g = g.reindex([o for o in order if o in g.index])
    g["non_carriers"] = g["N"] - g["carriers"]
    g["carrier_rate_%"] = (g["carriers"] / g["N"] * 100).round(2)
    return g[["carriers", "non_carriers", "N", "carrier_rate_%"]].astype(
        {"carriers": int, "non_carriers": int, "N": int})

cohorts = [c for c in ["LongNext", "PD-Frontline", "PPMI"] if c in cc["cohort"].unique()]
groups  = ["PD", "Controls", "Other"]

by_cxg = carrier_tab(cc["cohort"] + "-" + cc["group"], [f"{c}-{g}" for c in cohorts for g in groups])
by_grp = carrier_tab(cc["group"], groups)                 # pooled across cohorts
by_coh = carrier_tab(cc["cohort"], cohorts)               # pooled across groups
grand  = carrier_tab(pd.Series("ALL", index=cc.index), ["ALL"])

print("=== carriers by cohort x group ==="); print(by_cxg.to_string())
print("\n=== carriers by group (all cohorts) ==="); print(by_grp.to_string())
print("\n=== carriers by cohort (all groups) ==="); print(by_coh.to_string())
print("\n=== grand total ==="); print(grand.to_string())

out = pd.concat([
    by_cxg,
    by_grp.rename(index=lambda x: f"TOTAL-{x}"),
    by_coh.rename(index=lambda x: f"{x}-ALL"),
    grand,
])
out.to_csv("gba_carrier_counts_by_cohort_group.csv")
print("\nSaved -> gba_carrier_counts_by_cohort_group.csv")

## Best probe per variant

In [ ]:
# Ranking: highest sensitivity, then fewest FP, then most coverage, then fewest missing.
# Dead-orientation probes (sens=0) automatically lose to their working sibling.
pm = probe_metrics.copy()
pm["n_eval"] = pm[["TP", "FP", "TN", "FN"]].sum(axis=1)

best_probe = (pm.sort_values(
                ["variant_name", "sensitivity", "FP", "n_eval", "missing"],
                ascending=[True, False, True, False, True],
                na_position="last")
              .groupby("variant_name", as_index=False).first())

cols = ["variant_name", "probe_id", "pos", "REF", "ALT",
        "TP", "FP", "TN", "FN", "missing", "n_truth_pos",
        "sensitivity", "specificity", "precision"]
best_probe = best_probe[cols]

# --- Append truth variants that have no probe at all ---
truth_var_counts = truth_u["truth_tokens"].explode().dropna().value_counts()
probed_vars   = set(best_probe["variant_name"])
no_probe_vars = [v for v in truth_var_counts.index if v not in probed_vars]

if no_probe_vars:
    no_probe_rows = pd.DataFrame({
        "variant_name": no_probe_vars,
        "probe_id":     "(no probe for this variant)",
        "n_truth_pos":  [int(truth_var_counts[v]) for v in no_probe_vars],
    })
    # all metric columns left as NaN (not evaluated)
    best_probe = pd.concat([best_probe, no_probe_rows], ignore_index=True)
    print(f"Variants in truth with no probe ({len(no_probe_vars)}): {no_probe_vars}")

best_probe = best_probe.sort_values(
    ["n_truth_pos", "variant_name"], ascending=[False, True]).reset_index(drop=True)

n_probed = best_probe["probe_id"].ne("(no probe for this variant)").sum()
print(f"\nBest-probe table: {len(best_probe)} variants "
      f"({n_probed} with a probe, collapsed from {pm['probe_id'].nunique()} probes; "
      f"{len(best_probe) - n_probed} with no probe)")
print(best_probe.round(3).to_string(index=False))
best_probe.to_csv("gba_best_probe_per_variant.csv", index=False)
print("\nSaved → gba_best_probe_per_variant.csv")

## Calculating whole true positives and true negatives

In [ ]:
# FP / FN reconciliation across the whole dataset
variants_with_probe = set(genotypes["variant_name"].dropna().unique())

# Every truth carrier instance (one row per sample per variant they carry)
carrier_inst = (truth_u[[SAMPLE_ID_COL, "cohort", "truth_tokens"]]
                .explode("truth_tokens")
                .dropna(subset=["truth_tokens"])
                .rename(columns={SAMPLE_ID_COL: "sample_id",
                                 "truth_tokens": "variant_name"}))

# Lookups from the scored calls
detected   = set(map(tuple, valid.loc[valid.call_class == "TP",
                                       ["sample_id", "variant_name"]].values))
has_data   = set(map(tuple, valid[["sample_id", "variant_name"]].values))  # any non-missing call

def classify(row):
    key = (row.sample_id, row.variant_name)
    if row.variant_name not in variants_with_probe:
        return "FN_no_probe"
    if key in detected:
        return "detected"
    if key in has_data:
        return "FN_genuine"          # probe + data present, array called negative
    return "FN_no_data"              # probe exists but no non-missing call for this sample

carrier_inst["status"] = carrier_inst.apply(classify, axis=1)

print(f"Total truth carrier instances (sample × variant): {len(carrier_inst)}")
print("\n=== Detection breakdown ===")
print(carrier_inst["status"].value_counts().to_string())

fn = carrier_inst[carrier_inst.status.str.startswith("FN")]
print(f"\n--- FALSE NEGATIVES: {len(fn)} total ---")
print(f"  no probe for variant:          {(fn.status=='FN_no_probe').sum()}")
print(f"  probe present, no data/sample: {(fn.status=='FN_no_data').sum()}")
print(f"  genuine (probe+data, missed):  {(fn.status=='FN_genuine').sum()}")

print("\nGenuine FN detail:")
print(fn[fn.status=='FN_genuine'][["sample_id","cohort","variant_name"]].to_string(index=False))
print("\n'No probe' FN by variant:")
print(fn[fn.status=='FN_no_probe']["variant_name"].value_counts().to_string())

# False positives
fp_rows = valid[valid.call_class == "FP"]
fp_events = set(map(tuple, fp_rows[["sample_id", "variant_name"]].values))
print(f"\n--- FALSE POSITIVES ---")
print(f"  FP probe-calls (per probe):        {len(fp_rows)}")
print(f"  unique FP events (sample×variant): {len(fp_events)}")
print("\nFP probe-calls by variant:")
print(fp_rows["variant_name"].value_counts().to_string())

carrier_inst.to_csv("gba_carrier_reconciliation.csv", index=False)
print("\nSaved → gba_carrier_reconciliation.csv")

## Checking accuracy for hom N409S

In [ ]:
VARIANT = "N409S"   # change to inspect any variant

# Zygosity from truth (count how many times the token appears in the call string)
def allele_count(call, v):
    return [t.strip() for t in str(call).split("/")].count(v)

z = truth_u[[SAMPLE_ID_COL, TRUTH_CALL_COL]].copy()
z["n"] = z[TRUTH_CALL_COL].apply(lambda c: allele_count(c, VARIANT))
hom_samples = set(z.loc[z.n >= 2, SAMPLE_ID_COL])
het_samples = set(z.loc[z.n == 1, SAMPLE_ID_COL])
print(f"{VARIANT} in truth: {len(hom_samples)} homozygous, {len(het_samples)} heterozygous\n")

# Array calls at every probe for this variant
g = genotypes[genotypes["variant_name"] == VARIANT].copy()
g["truth_zyg"] = g["sample_id"].map(
    lambda s: "hom" if s in hom_samples else ("het" if s in het_samples else "non-carrier"))

# How each probe calls the truth HOMOZYGOTES (0=ref, 1=het, 2=hom-alt, NaN=missing)
hom_g = g[g["truth_zyg"] == "hom"]
print(f"=== Array call for the {len(hom_samples)} truth homozygotes, by probe ===")
print("(2 = correctly called homozygous · 1 = called het · 0 = called ref · NaN = missing)\n")
tab = (pd.crosstab(hom_g["probe_id"], hom_g["array_call"], dropna=False)
         .reindex(columns=[0, 1, 2, pd.NA], fill_value=0))
tab.columns = ["called_ref(0)", "called_het(1)", "called_hom(2)", "missing"]
tab["n_hom_samples"] = tab.sum(axis=1)
tab["pct_called_hom"] = (tab["called_hom(2)"] / tab["n_hom_samples"] * 100).round(1)
print(tab.sort_values("called_hom(2)", ascending=False).to_string())

# Same view for heterozygotes, as a contrast (a good probe calls these 1, not 2)
het_g = g[g["truth_zyg"] == "het"]
if len(het_g):
    print(f"\n=== For contrast: same probes on the {len(het_samples)} truth heterozygotes ===")
    htab = (pd.crosstab(het_g["probe_id"], het_g["array_call"], dropna=False)
              .reindex(columns=[0, 1, 2, pd.NA], fill_value=0))
    htab.columns = ["called_ref(0)", "called_het(1)", "called_hom(2)", "missing"]
    print(htab.to_string())

## sanity check to see if any probes in NBA need to be added to the tuple with variant names (It's ok to leave intronic variants we are not interested in, but these will not be accounted for in the false positives)

In [ ]:
unnamed = genotypes[genotypes["variant_name"].isna()]
print(f"Unnamed GBA-region probes: {unnamed['probe_id'].nunique()}")
print(f"Genotype rows on unnamed probes: {len(unnamed)}")

# of those, how many are non-ref calls (would-be carrier calls)?
nonref = unnamed[unnamed["array_call"] >= 1]
print(f"Non-ref calls on unnamed probes: {len(nonref)} "
      f"across {nonref['probe_id'].nunique()} probes, {nonref['sample_id'].nunique()} samples")
print("\nUnnamed probes with the most non-ref calls:")
print(nonref.groupby(["probe_id","pos"]).size()
            .sort_values(ascending=False).head(50).to_string())

## to get full list of probes across all fileset

In [ ]:
unique_probes = (all_gba_probes_df
                 .drop_duplicates(subset="ID")
                 .sort_values("POS")
                 .reset_index(drop=True))

unique_probes.to_csv("gba_all_unique_probes.csv", index=False)
print(f"{len(unique_probes)} unique probes → gba_all_unique_probes.csv")

## To get false positive calls per variant per cohort

In [ ]:
fp_rows = valid[valid.call_class == "FP"]

for v in ["K13R", "L28W", "T408M","E365K"]:
    sub = fp_rows[fp_rows.variant_name == v]
    print(f"=== {v}: {len(sub)} FP probe-calls ===")
    print(sub["cohort"].value_counts().to_string())
    print()

## Sample-level call table: reference sequencing (per cohort) vs the NBA call.

In [ ]:
SEQ_COLS = {"PD-Frontline": "PDF_sequencing",
            "LongNext":     "PDGEN_sequencing",
            "PPMI":         "PPMI_sequencing"}

seq_wide = (truth.drop_duplicates([SAMPLE_ID_COL, "cohort"])
                 .pivot(index=SAMPLE_ID_COL, columns="cohort", values=TRUTH_CALL_COL)
                 .rename(columns=SEQ_COLS)
                 .reindex(columns=list(SEQ_COLS.values())))

# NBA call written in the same "V1/V2" style as the truth calls, taking the best
# probe per variant so a variant with several probes is only counted once.
selected = set(best_probe.loc[best_probe["probe_id"] != "(no probe for this variant)",
                              "probe_id"])
nba = geno[geno["probe_id"].isin(selected)]

alt = nba[nba["array_call"] >= 1].copy()
alt["n_alt"] = alt["array_call"].astype(int)          # hom-alt (2) -> token written twice
nba_call = (alt.loc[alt.index.repeat(alt["n_alt"])]
               .sort_values(["sample_id", "variant_name"])
               .groupby("sample_id")["variant_name"].agg("/".join))

called = set(nba.loc[nba["array_call"].notna(), "sample_id"])   # ≥1 non-missing probe

sample_calls = seq_wide.reindex(sorted(truth_u[SAMPLE_ID_COL])).reset_index()
sample_calls["NBA_call"] = np.where(
    sample_calls[SAMPLE_ID_COL].isin(called),
    sample_calls[SAMPLE_ID_COL].map(nba_call).fillna("0"),   # genotyped, no alt allele
    "no call")                                               # no usable array data
sample_calls = sample_calls.fillna("")                       # sample not in that cohort

print(f"Samples in table: {len(sample_calls)}")
print(f"Sequenced in >1 cohort: "
      f"{(sample_calls[list(SEQ_COLS.values())].ne('').sum(axis=1) > 1).sum()}")
print(f"NBA: {(sample_calls['NBA_call'] == '0').sum()} reference, "
      f"{(~sample_calls['NBA_call'].isin(['0', 'no call'])).sum()} carriers, "
      f"{(sample_calls['NBA_call'] == 'no call').sum()} with no usable call")

sample_calls.to_csv("gba_sample_level_calls.csv", index=False)
print("\nSaved -> gba_sample_level_calls.csv")